## Minimum Distance Classifiers
### Consider the cork stoppers.xls data set containing measurements performed automatically by an image processing system on 150 cork stoppers(see Figure 2) belonging to three classes (ω1 - Super, ω2 - Average and ω3 Poor).

In your exercise consider only the classes ω1 and ω2, and a two-dimensional
space spanned by N and PRT/10.

• Find the Euclidean Linear Discriminants g1(x) and g2(x).

• Find the equation of the decision hyperplane, i.e., when g1(x) = g2(x).

• Compute the sensitivity, specificity, precision, F1Score and accuracy.

• Represent geometrically the patterns, class means, and decision hyperplane. You should obtain a plot like the one presented in Figure 3

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_excel("D:/UC_library/Mestrado/1_ANO/2_semestre/RP/PL/data_sets/CORK_STOPPERS.XLS",
                   sheet_name = "Data", 
                   usecols=["C","N.1","PRT"]) # usecols=["C","ART","N.1","PRT","ARM","PRM","ARTG","NG","PRTG","RAAR","RAN"]
df["PRT"] = df["PRT"] / 10
df = df[df["C"].isin([1, 2])]
df = pd.DataFrame(df.dropna())
quality = ["Super", "Average"]
color = df["C"].map({1 : quality[0], 2 : quality[1]})

x_1 = df[df["C"] == 1][['N.1', 'PRT']]
x_2 = df[df["C"] == 2][['N.1', 'PRT']]

mu_1 = np.array(np.mean(x_1, axis=0)).T
mu_2 = np.array(np.mean(x_2, axis=0)).T

print('g1(x) = ' + str(mu_1[0]) + 'x1 + ' + str(mu_1[1]) + 'x2 + ' + str(0.5 * np.matmul(mu_1, mu_1)))
print('g2(x) = ' + str(mu_2[0]) + 'x1 + ' + str(mu_2[1]) + 'x2 + ' + str(0.5 * np.matmul(mu_2, mu_2)))

w = mu_1 - mu_2 
b = -0.5*(w).T@(mu_1 + mu_2)  # '@' is equivalent to 'np.matmul' <- scalar product

print('\nDecision function (g1(x) - g2(x)):\n ' + str(w[0]) + 'x1 + ' + str(w[1]) + 'x2 = ' + str(b))

## Comupute specificity, sensitivity, precision, f1 score
x1 = df['N.1']
x2 = df["PRT"]

x = [np.array(x1), np.array(x2)]
decision = np.matmul(w.T, x) + b

predict = np.ones(np.shape(decision)[0])
predict[decision<0] = 2
# print(len(df['C']))

TP = TN = FP = FN = 0
for i  in range(0,len(df['C'])):
    if(df['C'].iloc[i] == predict[i] and df['C'].iloc[i] == 1):
        TP+=1
    if(df['C'].iloc[i] == predict[i] and df['C'].iloc[i] == 2):
        FN += 1
    if(df['C'].iloc[i] != predict[i] and df['C'].iloc[i] == 1):
        FP += 1
    if(df['C'].iloc[i] != predict[i] and df['C'].iloc[i] == 2):
        TN += 1
if(TP+TN+FP+FN != len(df['C'])): print(f"The sum of TP+TN+FN+FP != {len(df['C'])}")

sensitivity = TP/(TP + FN) * 100
specificity = TN/(TN + FP) * 100
precision = TP/(TP + FP) * 100
F1score = 2 * (sensitivity * specificity) / (sensitivity + specificity)

print(f'Sensitivity: {sensitivity}% \nSpecificity: {specificity}% \nPrecision: {precision}% \nF1score: {F1score}%')

fig = px.scatter(x=x1, y=x2, color=color, labels=dict(x = 'N', y = 'PRT'))

# Draw a line
fig.add_shape(type='line', x0=mu_1[0], y0=mu_1[1], x1=mu_2[0], y1=mu_2[1],
              line=dict(color = 'black', dash = 'dash'))

# Draw a line
fig.add_shape(type='line', x0=mu_1[0], y0=mu_1[1], x1=mu_2[0], y1=mu_2[1],
              line=dict(color = 'black', dash = 'dash'))

# draw a marker
fig.add_trace(go.Scatter(x = [mu_1[0]],
                         y = [mu_1[1]],
                         mode='markers',
                         marker=dict(
                             symbol = 'x',
                             size = 15,
                             color = 'green',
                             line = dict(width = 2) 
                         ),
                         name='mu_1'))
# Draw a marker
fig.add_trace(go.Scatter(x = [mu_2[0]],
                         y = [mu_2[1]],
                         mode='markers',
                         marker=dict(
                             symbol = 'x',
                             size = 15,
                             color = 'green',
                             line = dict(width = 2) 
                         ),
                         name='mu_2'))
fig.update_traces(marker_size = 8)

fig.show()


g1(x) = 55.28x1 + 36.504x2 + 2194.210208
g2(x) = 79.74x1 + 64.284x2 + 5245.450128

Decision function (g1(x) - g2(x)):
 -24.459999999999994x1 + -27.78000000000001x2 = 3051.23992
Sensitivity: 51.85185185185185% 
Specificity: 57.89473684210527% 
Precision: 84.0% 
F1score: 54.706927175843695%


## Repeat the process with Mahalanobis Linear Discriminants

In [5]:
# Standardize data
x1_norm = (x1 - np.mean(x1)) / np.std(x1, axis=0)
x2_norm = (x2 - np.mean(x2)) / np.std(x2, axis=0)

px.scatter(x = x1_norm, y=x2_norm, color=color)


cov_matrix = np.cov(pd.DataFrame([x1_norm, x2_norm]))

# We need to calculate the covariance matrix adn its inverse to apply to the decision function
C = np.linalg.inv(cov_matrix)
disc = np.matmul(w.T,C)
b1 = np.matmul(disc, -0.5*(mu_1 + mu_2))
decision_mahalanobis = np.matmul(disc ,x) + b1

predict = np.ones(np.shape(decision)[0])
predict[decision_mahalanobis<0] = 2

#_______________________Form here it is the same as above___________________#
TP = TN = FP = FN = 0
for i  in range(0,len(df['C'])):
    if(df['C'].iloc[i] == predict[i] and df['C'].iloc[i] == 1):
        TP+=1
    if(df['C'].iloc[i] == predict[i] and df['C'].iloc[i] == 2):
        FN += 1
    if(df['C'].iloc[i] != predict[i] and df['C'].iloc[i] == 1):
        FP += 1
    if(df['C'].iloc[i] != predict[i] and df['C'].iloc[i] == 2):
        TN += 1
if(TP+TN+FP+FN != len(df['C'])): print(f"The sum of TP+TN+FN+FP != {len(df['C'])}")

sensitivity = TP/(TP + FN) * 100
specificity = TN/(TN + FP) * 100
precision = TP/(TP + FP) * 100
F1score = 2 * (sensitivity * specificity) / (sensitivity + specificity)

print(f'Sensitivity: {sensitivity}% \nSpecificity: {specificity}% \nPrecision: {precision}% \nF1score: {F1score}%')

fig = px.scatter(x=x1, y=x2, color=color, labels=dict(x = 'N', y = 'PRT'))

# Draw a line
fig.add_shape(type='line', x0=mu_1[0], y0=mu_1[1], x1=mu_2[0], y1=mu_2[1],
              line=dict(color = 'black', dash = 'dash'))

# draw a marker
fig.add_trace(go.Scatter(x = [mu_1[0]],
                         y = [mu_1[1]],
                         mode='markers',
                         marker=dict(
                             symbol = 'x',
                             size = 15,
                             color = 'green',
                             line = dict(width = 2) 
                         ),
                         name='mu_1'))
# Draw a marker
fig.add_trace(go.Scatter(x = [mu_2[0]],
                         y = [mu_2[1]],
                         mode='markers',
                         marker=dict(
                             symbol = 'x',
                             size = 15,
                             color = 'green',
                             line = dict(width = 2) 
                         ),
                         name='mu_2'))

fig.show()

Sensitivity: 53.333333333333336% 
Specificity: 80.0% 
Precision: 96.0% 
F1score: 64.0%


If we compare the metrics we can say that the Mahalanobis aproach is more suitable for this case

## FISHER Linear Discriminant
### 1. Consider again Cork-Stoppers dataset and the two-dimensional input space defined by N and PRT/10.

• Generate the training and testing data sets by selecting randomly
70% of the samples for training and the remaining 30% for testing.

• Create a linear classifier based on LDA and on the selected training
data.

• Plot the training and testing data and the computed hyperplane.

• Compute the sensitivity, specificity, precision, F1Score and accuracy
on the training and testing sets.

In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

cork = pd.read_excel("D:/UC_library/Mestrado/1_ANO/2_semestre/RP/PL/data_sets/CORK_STOPPERS.XLS",
                   sheet_name = "Data", 
                   usecols=["C","N.1","PRT"]) # usecols=["C","ART","N.1","PRT","ARM","PRM","ARTG","NG","PRTG","RAAR","RAN"]
cork["PRT"] = cork["PRT"] / 10
# df = df[df["C"].isin([1, 2])]
cork = cork.dropna()
x = cork.to_numpy()[:,1:].astype(float)
y = cork.to_numpy()[:,0]


X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.3)


mu_N = np.mean(X_train, axis = 0)
mu_PRT = np.mean(Y_train, axis = 0)

print(np.shape(X_train))


(105, 2)
